# TOP GENES RARE VARIANT POLYGENIC RISK SCORE 
This script generates a rare variant polygenic risk score file for the top three significant genes identified in the SAIGE-GENE+ set-based analysis. The resulting file contains one column per gene, representing the rare-variant polygenic risk score burden for that gene for each individual. Each predicted deleterious rare variant (missense or loss-of-function; MAF < 1%) present within a gene for a given individual is assigned a weight of 1. The gene-level score is then computed as the sum of all predicted deleterious rare variants present within that gene for each individual.

In [ ]:
# Load packages 
import re
import os
import gc
import glob
import pandas as pd 
from functools import reduce

# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
# Remove burden polygenic risk score files 
!rm -rf *prs.tsv

# Create directory for gene lists 
!rm -rf gene_list
!mkdir -p gene_list 

# Create directory for burden scores 
!mkdir -p burden_sum_scores

In [ ]:
%%bash 
# Set workspace bucket containing the gene burden sum scores 
my_bucket="gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe"

# Create directory for gene burden sum scores 
mkdir -p burden_scores 

# Iterate through autosomes (chromosomes 1–22)
for chrom in {1..22}; do

  # Iterate through the three genes of interest (top genes identified in the association analyses)  
  for gene in ENSG00000119772 ENSG00000168769 ENSG00000161547; do
  
  # Create a chromosome-specific directory to organize the downloaded burden score files
  mkdir -p burden_scores/burden_scores_chr${chrom}
  
  # Download the gene-level burden score file for the current gene and chromosome
  # Files are retrieved from the corresponding chromosome-specific directory in the workspace bucket
  gsutil cp "${my_bucket}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chrom}/*${gene}-chr${chrom}.sscore" burden_scores/burden_scores_chr${chrom}
  done
done 

In [ ]:
# Print chromosome directories for burden sum scores 
!ls burden_scores

In [ ]:
# Gene top genes discovered in set-based gene associations 
genes = pd.DataFrame({"gene_id" : ["ENSG00000119772", "ENSG00000168769", "ENSG00000161547"]})

In [ ]:
# Initialize the overall merged master dataframe
merged_master = None 

# Iterate through autosomes (chromosomes 1–22)
for chrom in range(1, 23):
    
    print(f"Processing chromosome {chrom} ....")   
    
    # Define the directory containing gene-level burden score files for the current chromosome
    folder_path = f"burden_scores/burden_scores_chr{chrom}"
    # Retrieve and sort all files within the chromosome-specific directory
    files = sorted(glob.glob(os.path.join(folder_path, "*")))

    # Skip the chromosome if no burden score files are available
    if not files:
        print(f"No files found for chromosome {chrom}, skipping...")
        continue
    
    # Initialize the chromosome-specific master dataframe using individual identifiers
    # (FID and IID) from the first burden score file
    master = pd.read_csv(files[0], sep="\t")[["#FID", "IID"]]
    
    # Iterate through the burden score files for the current chromosome
    for filename in files:
        
        # Extract the Ensembl gene identifier (ENSG ID) from the filename
        match = re.search(r"(ENSG\d+)", filename)
        
        # Skip files that do not contain a recognizable ENSG identifier
        if not match:
            continue
        
        # Extract the gene ENSG ID from the matched filename
        ensg_id = match.group(1)

        # Retain only genes included in the predefined list of top genes
        if ensg_id not in genes["gene_id"].unique():
            continue
        
        # Read the gene-level burden score file
        df = pd.read_csv(filename, sep="\t")
        
        # Verify that NAMED_ALLELE_DOSAGE_SUM and SCORE1_SUM contain
        # identical values across all individuals
        # Skip the file if the two dosage score columns do not match
        if not (df["NAMED_ALLELE_DOSAGE_SUM"] == df["SCORE1_SUM"]).all():
            continue
      
        print(f"Processing gene {ensg_id} ....")

        # Retain individual identifiers and the gene-level burden score
        # Rename SCORE1_SUM using the ENSG ID to identify the corresponding gene
        df = df[["#FID", "IID", "SCORE1_SUM"]].rename(
                columns={"SCORE1_SUM": f"{ensg_id}_SCORE1_SUM"}
            )
        
        # Merge the gene-level burden score into the chromosome-specific master dataframe
        # using FID and IID to match individuals
        # An outer join retains individuals present in either dataframe
        master = pd.merge(master, df, on=["#FID", "IID"], how="outer")
        
    # Merge the chromosome-specific master dataframe into the overall dataset
    # An outer join retains individuals across all processed chromosomes
    if merged_master is None:
        merged_master = master
    else:
        merged_master = pd.merge(merged_master, master, on=["#FID", "IID"], how="outer")
    
    print(f"Chromosome {chrom} merge complete. Master shape: {master.shape}")

# Confirm completion of the chromosome-level merging procedure
print("All chromosomes merged.")
print("Final shape:", merged_master.shape)

# Export the merged gene-level rare-variant PRS dataframe as a tab-delimited file
merged_master.to_csv(f"top_genes_prs.tsv", sep ='\t', index=False)

# Clean up memory
del master, merged_master
gc.collect()


In [ ]:
# Copy top gene polygenick risk score file to workspace bucket 
!gsutil cp top_genes_prs.tsv {my_bucket}/data/gene_burden_analysis/burden_gene_sets/top_genes/ 